# 02 — Sessions and Dataset (Ahmed, S1.8)

## Plan

1. Load the project configuration.
2. List the session-selection rules, read from `config.yaml`.
3. Show the 10 selected sessions and the reserves from `sessions.yaml`, and check them against the rules.
4. Show the size of the frozen release and how many sessions each selection rule removed.
5. Explain how one ready session file is built, and show the shape of every real file in `data/processed/`.

**Source of truth:** `config.yaml`, `sessions.yaml`, the shared project helpers and the generated `data/processed/*.npz`. Code is not reimplemented in this notebook.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Open this notebook inside the brain-decoding repository.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.settings import (
    load_config, load_session_list, load_session_info,
    list_processed_sessions, load_session,
)

import pandas as pd
import numpy as np
import yaml

config = load_config()
print("Project:", config["project_name"])
print("Release freeze:", config["ibl"]["release_freeze"])
print("Aggregate tag:", config["ibl"]["aggregate_tag"])

Project: brain-decoding
Release freeze: 2023_12_bwm_release
Aggregate tag: 2024_Q2_IBL_et_al_BWM


## 1. Session selection criteria

The project uses the IBL Brain-Wide Map freeze set in `config.yaml`. A session must pass IBL's official trial rules, have enough good neurons and enough brain regions with enough neurons, and the selection keeps one session per mouse and at most two per lab. Sessions with the most neurons are taken first; the next ones that pass every rule are kept as reserves.

Section 3 below recomputes how many sessions each of these rules removed.

In [2]:
d = config["data"]
criteria = {
    "Target sessions": d["n_sessions"],
    "Minimum eligible neurons": d["min_neurons"],
    "Minimum eligible trials": d["min_trials"],
    "Minimum error trials": d["min_errors"],
    "Minimum qualifying brain regions": d["min_regions"],
    "Minimum neurons per qualifying region": d["neurons_per_region"],
    "Maximum sessions per lab": d["max_sessions_per_lab"],
    "Good neuron QC": d["good_neuron_qc"],
    "Region mapping": d["region_mapping"],
    "Excluded regions": ", ".join(d["excluded_regions"]),
    "Feature window (seconds)": str(d["window"]),
}
display(pd.DataFrame(criteria.items(), columns=["Selection criterion", "Configuration value"]))

,Selection criterion,Configuration value
0,Target sessions,10
1,Minimum eligible neurons,60
2,Minimum eligible trials,400
3,Minimum error trials,3
4,Minimum qualifying brain regions,3
5,Minimum neurons per qualifying region,10
6,Maximum sessions per lab,2
7,Good neuron QC,1.0
8,Region mapping,Beryl
9,Excluded regions,"root, void"


## 2. Selected sessions and reserves

`src/data/sessions.py` owns the selection logic and writes `sessions.yaml`. The next cells inspect that file without downloading data or modifying it.

In [3]:
with (ROOT / "sessions.yaml").open(encoding="utf-8") as f:
    selection = yaml.safe_load(f)
selected = selection.get("sessions", [])
reserves = selection.get("reserves", [])

rows = []
for s in selected:
    rows.append({
        "session_id": s["session_id"],
        "subject": s["subject"],
        "lab": s["lab"],
        "n_pids": len(s["pids"]),
        "n_neurons": s["n_neurons"],
        "n_regions": s["n_regions"],
    })
summary = pd.DataFrame(rows)
print("Selected sessions:", len(selected))
print("Reserve sessions:", len(reserves))
display(summary)

Selected sessions: 10
Reserve sessions: 14


,session_id,subject,lab,n_pids,n_neurons,n_regions
0,746d1902-fa59-4cab-b0aa-013be36060d5,ZFM-01592,mainenlab,2,429,17
1,ee40aece-cffd-4edb-a4b6-155f158c666a,ZM_2241,mainenlab,2,427,15
2,f10efe41-0dc0-44d0-8f26-5ff68dca23e9,CSHL060,churchlandlab,2,396,9
3,56b57c38-2699-4091-90a8-aba35103155e,SWC_054,mrsicflogellab,2,391,13
4,4d8c7767-981c-4347-8e5e-5d5fffe38534,SWC_038,mrsicflogellab,2,389,10
5,6274dda8-3a59-4aa1-95f8-a8a549c46a26,CSH_ZAD_025,zadorlab,2,349,10
6,2bdf206a-820f-402f-920a-9e86cd5388a4,UCLA037,churchlandlab_ucla,2,346,9
7,8207abc6-6b23-4762-92b4-82e05bed5143,CSH_ZAD_024,zadorlab,2,332,8
8,dfbe628d-365b-461c-a07f-8b9911ba83aa,KS046,cortexlab,2,331,7
9,09394481-8dd2-4d5c-9327-f2753ede92d7,ibl_witten_26,wittenlab,2,326,12


In [4]:
# Checks available from the saved selection metadata.
assert len(selected) == d["n_sessions"], "Wrong number of selected sessions"
assert summary["subject"].nunique() == len(selected), "Repeated subject"
assert (summary["lab"].value_counts() <= d["max_sessions_per_lab"]).all(), "Lab limit exceeded"
assert (summary["n_neurons"] >= d["min_neurons"]).all(), "Neuron minimum not met"
assert (summary["n_regions"] >= d["min_regions"]).all(), "Region count minimum not met"
assert all(len(s["pids"]) > 0 for s in selected), "A session has no probe IDs"
print("PASS: saved session count, subject uniqueness, lab limits and recorded neuron/region counts")
print("Trial counts per session are rechecked against IBL in scripts/stage1_check.py.")

PASS: saved session count, subject uniqueness, lab limits and recorded neuron/region counts
Trial counts per session are rechecked against IBL in scripts/stage1_check.py.


## 3. Size of the release and how many sessions each rule removed

`bwm_query` lists the frozen release. `run_selection_audit` in `src/data/sessions.py` downloads IBL's two aggregate tables (the same ones the selection used) and applies the rules again, in the same order, counting the sessions left after each one. It is read-only: it uses the saved `sessions.yaml` only to tell a session whose trials failed to load apart from one removed by the mouse or lab rule, and it never changes the selection. It raises an error if the rules do not give back exactly the saved selected sessions and reserves.

In [5]:
from brainwidemap import bwm_query
from src.common.settings import connect_ibl
from src.data.sessions import run_selection_audit

release = bwm_query(freeze=config["ibl"]["release_freeze"])
print("Sessions in the release:", release["eid"].nunique())
print("Probe insertions:", release["pid"].nunique())
print("Mice:", release["subject"].nunique(), "| Labs:", release["lab"].nunique())

one = connect_ibl()
audit = run_selection_audit(one, config)
display(audit)
assert audit["n_sessions"].iloc[-1] == len(selected) + len(reserves)
print("Selected:", len(selected), "| Reserves:", len(reserves))

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
Sessions in the release: 459
Probe insertions: 699
Mice: 139 | Labs: 12


Loading BWM release...
Loading bwm_query results from fixtures/2023_12_bwm_release.csv
Available sessions: 459


D:\projects\brain-decoding\.venv\Lib\site-packages\one\webclient.py:1053: RuntimeWarning: The shared public account will be retired soon. Register your own account at https://openalyx.internationalbrainlab.org/signup
  emit(message)


(S3) D:\projects\brain-decoding\data\raw\bwm_tables\clusters.pqt:   0%|          | 0.00/91.6M [00:00<?, ?B/s]

(S3) D:\projects\brain-decoding\data\raw\bwm_tables\clusters.pqt:   0%|          | 262k/91.6M [00:01<08:43, 174kB/s]

(S3) D:\projects\brain-decoding\data\raw\bwm_tables\clusters.pqt:   1%|          | 1.05M/91.6M [00:01<01:49, 823kB/s]

... (198 outputs left out)

Filtering good neurons...


Filtering valid trials...


,step,n_sessions,n_removed
0,sessions in the frozen release,459,0
1,have good neurons outside root/void,459,0
2,"pass IBL trial rules, >= 400 trials, >= 3 errors",225,234
3,>= 60 good neurons,192,33
4,>= 3 regions with >= 10 neurons,171,21
5,one session per mouse,132,39
6,at most 2 sessions per lab,24,108
7,official trials load without an error,24,0


Selected: 10 | Reserves: 14


## 4. How one ready session is built

This is the implementation sequence specified by the team plan:

`load_session_info` → `load_trials` → `clean_trials` → `load_neurons` → window around `firstMovement_times` → `firing_rates` → `assemble_session` → `save_session`.

Each row of `X` is a trial; each column is a good neuron. Features are firing rates in Hz in the interval **[−0.1 s, 0.0 s] relative to first wheel movement**. The selected choice is `y=1` for `choice=+1` and `y=0` for `choice=−1`.

`build_dataset.py` is responsible for executing this process; the notebook only describes and inspects it. Its execution requires teammate-owned `clean_trials.py`, `neurons.py`, and `features.py` to be present and compatible.

In [6]:
expected_keys = {
    "X", "y", "trial_number", "stim_side", "contrast", "is_error",
    "is_zero_contrast", "prob_left", "reaction_time", "neuron_id",
    "neuron_region", "session_id",
}
ready = list_processed_sessions()
print("Processed sessions actually available:", len(ready), "of", len(selected))

processed_rows = []
for session_id in ready:
    data = load_session(session_id)
    keys = set(data)
    if keys != expected_keys:
        raise AssertionError(f"Unexpected keys for {session_id}: missing={expected_keys - keys}, extra={keys - expected_keys}")
    x = np.asarray(data["X"])
    assert x.ndim == 2 and np.isfinite(x).all(), f"Invalid X: {session_id}"
    assert x.shape[0] == len(data["y"]) and x.shape[1] == len(data["neuron_id"])
    processed_rows.append({
        "session_id": session_id,
        "n_trials": x.shape[0],
        "n_neurons": x.shape[1],
        "n_regions": len(set(map(str, data["neuron_region"]))),
        "X_shape": str(x.shape),
    })

if processed_rows:
    shapes = pd.DataFrame(processed_rows)
    display(shapes)
    # the neurons saved per session must equal the count recorded in sessions.yaml
    recorded = summary.set_index("session_id")["n_neurons"]
    assert (shapes.set_index("session_id")["n_neurons"] == recorded.reindex(shapes["session_id"])).all()
    print("Neurons in every ready file match n_neurons in sessions.yaml.")
else:
    print("No ready .npz files found. Shape tables will populate automatically after S1.7 succeeds.")

Processed sessions actually available: 10 of 10


,session_id,n_trials,n_neurons,n_regions,X_shape
0,746d1902-fa59-4cab-b0aa-013be36060d5,481,429,17,"(481, 429)"
1,ee40aece-cffd-4edb-a4b6-155f158c666a,527,427,15,"(527, 427)"
2,f10efe41-0dc0-44d0-8f26-5ff68dca23e9,429,396,9,"(429, 396)"
3,56b57c38-2699-4091-90a8-aba35103155e,415,391,13,"(415, 391)"
4,4d8c7767-981c-4347-8e5e-5d5fffe38534,562,389,10,"(562, 389)"
5,6274dda8-3a59-4aa1-95f8-a8a549c46a26,591,349,10,"(591, 349)"
6,2bdf206a-820f-402f-920a-9e86cd5388a4,427,346,9,"(427, 346)"
7,8207abc6-6b23-4762-92b4-82e05bed5143,491,332,8,"(491, 332)"
8,dfbe628d-365b-461c-a07f-8b9911ba83aa,516,331,7,"(516, 331)"
9,09394481-8dd2-4d5c-9327-f2753ede92d7,433,326,12,"(433, 326)"


Neurons in every ready file match n_neurons in sessions.yaml.


## Conclusions

- The frozen release `2023_12_bwm_release` has 459 sessions, 699 probe insertions, 139 mice and 12 labs.
- All 459 sessions have good neurons outside `root` and `void`, so that rule removed none.
- IBL's own trial rules together with our minimums (at least 400 trials and 3 error trials) removed the most on data quality: 234 sessions, leaving 225.
- At least 60 good neurons removed 33 more (192 left), and at least 3 regions with 10 or more neurons removed 21 more (171 left).
- One session per mouse removed 39 (132 left). At most 2 sessions per lab then removed 108, leaving 24. With 12 labs and a cap of 2, 24 is the most the rules can ever keep, so the lab cap is what decides the final pool.
- No session was lost because its trials failed to load.
- The 24 sessions are the 10 selected sessions (the ones with the most neurons) and the 14 reserves. The audit applies the rules again and gets back exactly the saved `sessions.yaml`, in the same order, so the saved selection really comes from these rules.
- All 10 ready files exist with the 12 required keys and no NaN in `X`. Their shapes go from 415 to 591 trials and from 326 to 429 neurons, and the neurons in every file match `n_neurons` in `sessions.yaml`, which was counted from IBL's own aggregate table.